In [1]:
from datasets import load_dataset
import pandas as pd


d:\Work\Github\google-tunix-kaggle\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# ============================================================================
# COMPETITION-ALIGNED BENCHMARK SUITE
# Prioritizes: Creative Writing, Ideation, Summarization
# Lower weight: Math, Code
# ============================================================================

# ============================================================================
# 1. CREATIVE WRITING - LitBench (Reddit WritingPrompts)
# ============================================================================
# 2,480 human-labeled story comparisons from Reddit
# Best judge: Claude 3.7 (73% human agreement)
try:
    litbench = load_dataset("VioletXF/LitBench", split="test")
    litbench_sample = litbench.shuffle(seed=42).select(range(100))
    litbench_df = pd.DataFrame({
        'domain': 'creative_writing',
        'input': [item['prompt'] for item in litbench_sample],
        'output': None,  # Use LLM-as-judge
        'reference_story': [item['story_a'] for item in litbench_sample],  # For comparison
        'benchmark': 'LitBench'
    })
except:
    print("LitBench not available - using alternative")
    litbench_df = pd.DataFrame()

# ============================================================================
# 2. CREATIVE WRITING - Alternative: EQ-Bench Creative Prompts
# ============================================================================
# 32 challenging creative writing prompts designed to expose model weaknesses
creative_prompts = [
    "Write a story about a librarian who discovers that books in the restricted section are gateways to other dimensions.",
    "Write a humorous story about a time traveler who keeps accidentally changing minor historical events.",
    "Write a suspenseful story about someone who realizes their memories are being slowly replaced.",
    "Write a story from the perspective of the last tree in a deforested world.",
    "Write a romantic story where two people fall in love through competing in a cooking competition.",
    "Write a story about an AI that becomes self-aware during a therapy session.",
    "Write a story where the protagonist discovers they've been living the same day for years without realizing it.",
    "Write a story about a detective who can see one hour into the future.",
    "Write a horror story set in an abandoned amusement park at midnight.",
    "Write a story about a musician who discovers their music can physically heal people.",
    "Write a story from the perspective of a house watching its families change over 100 years.",
    "Write a story about two strangers who swap bodies and must live each other's lives for a week.",
    "Write a story about a chef whose food makes people experience their happiest memory.",
    "Write a story set in a world where lying is physically impossible.",
    "Write a story about someone who wakes up with a completely different skill set every morning.",
    "Write a story about a parent explaining to their child why the stars are disappearing.",
    "Write a story where the narrator gradually realizes they're not human.",
    "Write a story about the friendship between a pessimist and an optimist during an apocalypse.",
    "Write a story from the perspective of the ocean watching humanity evolve.",
    "Write a story about someone who can taste emotions.",
]

creative_writing_df = pd.DataFrame({
    'domain': 'creative_writing',
    'input': creative_prompts,
    'output': None,  # LLM-as-judge evaluation
    'benchmark': 'Creative_Writing_Prompts'
})

# ============================================================================
# 3. SUMMARIZATION - CNN/DailyMail (200 samples)
# ============================================================================
# Standard summarization benchmark: 781 token articles → 56 token summaries
cnn_dm = load_dataset("cnn_dailymail", "3.0.0", split="test")
cnn_dm_sample = cnn_dm.shuffle(seed=42).select(range(200))
cnn_dm_df = pd.DataFrame({
    'domain': 'summarization',
    'input': [f"Summarize the following article:\n\n{item['article']}" for item in cnn_dm_sample],
    'output': [item['highlights'] for item in cnn_dm_sample],
    'benchmark': 'CNN_DailyMail'
})

# ============================================================================
# 4. SUMMARIZATION - XSum (100 samples) 
# ============================================================================
# Extreme summarization: 431 word articles → 23 word single-sentence summaries
# More abstractive than CNN/DM
# xsum = load_dataset("xsum", split="test")
# xsum_sample = xsum.shuffle(seed=42).select(range(100))
# xsum_df = pd.DataFrame({
#     'domain': 'summarization',
#     'input': [f"Write a single-sentence summary of this article:\n\n{item['document']}" for item in xsum_sample],
#     'output': [item['summary'] for item in xsum_sample],
#     'benchmark': 'XSum'
# })

# ============================================================================
# 5. CREATIVE IDEATION - Custom Benchmark
# ============================================================================
# No standard benchmark exists - create diverse ideation prompts
ideation_prompts = [
    "Generate 5 innovative business ideas that combine AI with sustainable agriculture.",
    "Brainstorm 7 creative ways to reduce food waste in restaurants.",
    "Come up with 5 unique mobile app concepts for improving mental health.",
    "Generate 6 innovative solutions for making public transportation more accessible.",
    "Brainstorm 5 creative marketing campaigns for a zero-waste grocery store.",
    "Come up with 7 innovative features for a smart home designed for elderly users.",
    "Generate 5 creative ways to teach mathematics to visual learners.",
    "Brainstorm 6 innovative uses for recycled ocean plastic.",
    "Come up with 5 unique team-building activities for remote workers.",
    "Generate 7 creative solutions for reducing noise pollution in cities.",
    "Brainstorm 5 innovative children's toys that teach coding concepts.",
    "Come up with 6 creative ways to preserve endangered languages.",
    "Generate 5 innovative concepts for community spaces that promote social connection.",
    "Brainstorm 7 creative approaches to making museums more engaging for children.",
    "Come up with 5 unique subscription box ideas for niche hobbies.",
    "Generate 6 innovative solutions for reducing single-use plastics in events.",
    "Brainstorm 5 creative ways to gamify daily exercise.",
    "Come up with 7 innovative features for an app that connects neighbors.",
    "Generate 5 creative approaches to teaching history through interactive experiences.",
    "Brainstorm 6 innovative solutions for making concerts accessible to deaf audiences.",
]

ideation_df = pd.DataFrame({
    'domain': 'creative_ideation',
    'input': ideation_prompts,
    'output': None,  # LLM-as-judge evaluation
    'benchmark': 'Creative_Ideation'
})

# ============================================================================
# 6. INSTRUCTION FOLLOWING - AlpacaEval 2.0 (200 samples)
# ============================================================================
# Covers diverse instructions including creative tasks
alpaca_eval = load_dataset("tatsu-lab/alpaca_eval", "alpaca_eval_gpt4_baseline", split="eval")
alpaca_sample = alpaca_eval.shuffle(seed=42).select(range(200))
alpaca_df = pd.DataFrame({
    'domain': 'instruction_following',
    'input': [item['instruction'] for item in alpaca_sample],
    'output': [item['output'] for item in alpaca_sample],  # GPT-4 baseline
    'benchmark': 'AlpacaEval'
})

# ============================================================================
# 7. MATH - GSM8K (100 samples) - LOWER PRIORITY
# ============================================================================
gsm8k = load_dataset("openai/gsm8k", "main", split="test")
gsm8k_sample = gsm8k.shuffle(seed=42).select(range(100))
gsm8k_df = pd.DataFrame({
    'domain': 'math',
    'input': [item['question'] for item in gsm8k_sample],
    'output': [item['answer'].split('####')[-1].strip() for item in gsm8k_sample],
    'benchmark': 'GSM8K'
})

# ============================================================================
# 8. CODE - HumanEval (50 samples) - LOWER PRIORITY
# ============================================================================
humaneval = load_dataset("openai/openai_humaneval", split="test")
humaneval_sample = humaneval.shuffle(seed=42).select(range(50))
humaneval_df = pd.DataFrame({
    'domain': 'code',
    'input': [item['prompt'] for item in humaneval_sample],
    'output': [item['canonical_solution'] for item in humaneval_sample],
    'benchmark': 'HumanEval'
})

# ============================================================================
# 9. SCIENCE - ARC-Challenge (50 samples)
# ============================================================================
arc = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")
arc_sample = arc.shuffle(seed=42).select(range(50))
arc_df = pd.DataFrame({
    'domain': 'science',
    'input': [
        f"Question: {item['question']}\n" + 
        "\n".join([f"{label}) {text}" for label, text in zip(item['choices']['label'], item['choices']['text'])])
        for item in arc_sample
    ],
    'output': [item['answerKey'] for item in arc_sample],
    'benchmark': 'ARC-Challenge'
})

# ============================================================================
# Combine with proper weighting for competition alignment
# ============================================================================
all_benchmarks = pd.concat([
    creative_writing_df,  # HIGH PRIORITY - 20 samples
    ideation_df,          # HIGH PRIORITY - 20 samples  
    cnn_dm_df,           # HIGH PRIORITY - 200 samples
    xsum_df,             # HIGH PRIORITY - 100 samples
    alpaca_df,           # HIGH PRIORITY - 200 samples (mixed domains)
    arc_df,              # MEDIUM - 50 samples
    gsm8k_df,            # LOWER - 100 samples
    humaneval_df,        # LOWER - 50 samples
], ignore_index=True)

# Save


LitBench not available - using alternative


RuntimeError: Dataset scripts are no longer supported, but found alpaca_eval.py

In [ ]:
all_benchmarks.to_parquet("/content/drive/MyDrive/kaggle-tunix-folder/dataset/competition_aligned_benchmarks.parquet")

print(f"✓ Created competition-aligned benchmark suite with {len(all_benchmarks)} questions")
print(f"\nBenchmark distribution:")
print(all_benchmarks['benchmark'].value_counts())
print(f"\nDomain distribution:")
print(all_benchmarks['domain'].value_counts())